# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row =** one `(client, content item)` pair, evaluated at a monthly anchor date. Every daily row for that pair inside a fixed **90-day trailing window** rolls up into one feature row; the **30 days immediately after** the anchor supplies the label — did the pair hold or grow, rather than decay.

**Table(s):** `fact_content_daily_performance` is the workhorse — daily grain, partitioned by `month=YYYY-MM`, one row per `(report_date, client, content item)`. `dim_content` and `dim_clients` are joined in for **context only** (content/client attributes, `gsc_data_start` / `ga4_data_start`).

**Time window:** trailing 90 days ending at the anchor = feature window; the 30 days right after = label window. For this notebook the anchor sits inside a mid-panel month, `month=2026-03`, so I iterate on real data without ever touching the sealed test month. The `_sample` file is June 2026 — the natural outcome window of any past→future label — so it stays sealed here, used only to sanity-check query mechanics later, never to build label logic.

**What I'd predict:** a binary momentum/recovery flag — did the pair's forward-30-day daily average hold at or above its trailing-90-day daily average (details in Part 3's label-derived column, deliberately isolated so it can be deleted).

**Verified below in Part 3.**

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Before sorting fields, I confirm the real column names for this month's partition — the dataset card documents table grains and row counts but not the literal column list, and the parquet schema sits behind the access gate. Run the cell below first; if a name in the table further down is stale, the `DESCRIBE` output is the source of truth, not this markdown.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

path_daily = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"

schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{path_daily}')").df()
print(schema_df.to_string(index=False))

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

| Field | Bucket | Why |
|---|---|---|
| `report_date` | context | Defines the daily grain and the window boundaries; not fed to the model as a raw value |
| `client_hash_id` | context | Pseudonymized join/identity key, not predictive on its own |
| `content_hash_id` | context | Pseudonymized join/identity key |
| `gsc_clicks` (90-day sum) | feature | Historical GSC performance up to the anchor date — known before the anchor, by definition |
| `gsc_impressions` (90-day sum) | feature | Same — purely historical |
| `ctr_avg_90d` (derived from `gsc_clicks`/`gsc_impressions`) | feature | Derived from clicks/impressions, both already historical |
| `gsc_avg_position` (90-day avg) | feature | Historical GSC ranking signal |
| `gsc_data_available` (boolean, or similarly named) | context / availability check | Used for the `IS TRUE` availability query in Part 3, not a training feature |
| forward-30-day clicks (derived, not a raw column) | label | Computed only from the 30 days *after* the anchor — exactly what must never leak into features |
| `fact_content_query_90d` (whole table) | excluded | Its own 90-day window is a fixed snapshot, not parameterized by our per-row anchor date — joining it in would silently mix a different, possibly overlapping window into the feature set. Excluded until window-alignment logic is built and checked, not because it's uninteresting |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries below, all on `month=2026-03`, then the five-feature frame, then the deliberate leak.

In [ ]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS n_distinct_keys
    FROM read_parquet('{path_daily}')
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_distinct_keys
0,9841378,9841378


In [ ]:
n_rows = int(grain_check['n_rows'][0])
n_keys = int(grain_check['n_distinct_keys'][0])
if n_rows == n_keys:
    print(f"Grain confirmed: {n_rows:,} rows, {n_keys:,} distinct (report_date, client, content) keys.")
else:
    print(f"Grain NOT confirmed: {n_rows:,} rows vs {n_keys:,} distinct keys — {n_rows - n_keys:,} duplicate key(s).")

Grain confirmed: 9,841,378 rows, 9,841,378 distinct (report_date, client, content) keys.


In [ ]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{path_daily}')
""").df()
span_check

,n_rows,first_date,last_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS n_gsc_available
    FROM read_parquet('{path_daily}')
""").df()

n_total = int(availability_check['n_total'][0])
n_avail = int(availability_check['n_gsc_available'][0])
print(f"{n_avail:,} of {n_total:,} rows ({n_avail / n_total:.1%}) have GSC data available in {MONTH}.")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3,611,061 of 9,841,378 rows (36.7%) have GSC data available in 2026-03.


,n_total,n_gsc_available
0,9841378,3611061


### Five features (built from `month=2026-03`, one line each: knowable at the decision moment because…)

1. **`clicks_90d`** (sum of clicks) — knowable because it's a running historical total; every value in the sum sits before the anchor date.
2. **`impressions_90d`** (sum of impressions) — same reasoning, purely historical.
3. **`ctr_avg_90d`** (mean CTR) — derived entirely from clicks/impressions already in the past.
4. **`position_avg_90d`** (mean ranking position) — a historical GSC signal, observed, not predicted.
5. **`active_days_90d`** (count of distinct dates with `gsc_available IS TRUE`) — a coverage feature; whether data existed for a day is itself known on that day, never inferred from the future.

In [ ]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)                                                   AS clicks_90d,
        SUM(gsc_impressions)                                              AS impressions_90d,
        SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)                      AS ctr_avg_90d,
        AVG(gsc_avg_position)                                             AS position_avg_90d,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_data_available IS TRUE) AS active_days_90d
    FROM read_parquet('{path_daily}')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"{len(features):,} (client, content) pairs in the {MONTH} feature frame.")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331,437 (client, content) pairs in the 2026-03 feature frame.


,client_hash_id,content_hash_id,clicks_90d,impressions_90d,ctr_avg_90d,position_avg_90d,active_days_90d
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2.0,1140.0,0.001754,4.394234,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,0.0,57.0,0.000000,2.714744,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,0.0,149.0,0.000000,6.481453,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,6.0,1421.0,0.004222,6.320337,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,16.0,2770.0,0.005776,4.459107,31


### The trap: one label-derived column, on purpose

For this week's toy demo the label is a stand-in — `clicks_90d` above (this month's) median, since the real forward-30-day label needs April data this notebook doesn't load. The mechanism is the same regardless of which month the label comes from: adding a column that's a direct function of the label collapses the model to (near-)perfect, which is the tell that it isn't learning anything — it's just reading the answer key back.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features["label_momentum"] = (features["clicks_90d"] > features["clicks_90d"].median()).astype(int)

honest_X = features[["impressions_90d", "ctr_avg_90d", "position_avg_90d", "active_days_90d"]].fillna(0)
y = features["label_momentum"]

X_train, X_test, y_train, y_test = train_test_split(honest_X, y, test_size=0.3, random_state=0)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

Honest AUC (no leak): 0.978


In [ ]:
leaky_X = honest_X.copy()
leaky_X["clicks_90d_leak"] = features["clicks_90d"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(leaky_X, y, test_size=0.3, random_state=0)
leak_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leak_auc = roc_auc_score(y_test_l, leak_model.predict_proba(X_test_l)[:, 1])
print(f"Leaky AUC (with clicks_90d_leak): {leak_auc:.3f}")
print("\n^ watch this jump toward 1.0 — that's the trap, not a win.")

Leaky AUC (with clicks_90d_leak): 1.000

^ watch this jump toward 1.0 — that's the trap, not a win.


In [ ]:
del leaky_X
print(f"Honest AUC: {honest_auc:.3f}  |  Leaky AUC (discarded): {leak_auc:.3f}")
print(f"Keeping {honest_auc:.3f} as the real, decision-supporting number.")

Honest AUC: 0.978  |  Leaky AUC (discarded): 1.000
Keeping 0.978 as the real, decision-supporting number.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
panel_depth = con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start,
        COUNT(*)            AS n_clients
    FROM read_parquet('{BASE}/dim_clients.parquet')
""").df()
panel_depth

,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start,n_clients
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01,104


- **Unbalanced panel:** `gsc_data_start` / `ga4_data_start` differ per client (see the spread above). A 90-day trailing window anchored close to a client's own start date will be thinner than one anchored later — the same window length doesn't mean the same amount of real history for every pair.
- **`fact_content_query_90d` stays excluded here:** its 90-day window is a fixed snapshot, not parameterized by our per-row anchor date. Combining it with the daily fact table without first checking window alignment risks silently mixing in a different time span.
- **Pseudonymization removes qualitative sanity-checking:** hash keys protect client/content identity, but that also means a weird number can't be eyeballed against "does this actually make sense for this kind of site" — only the statistics can be judged.
- **June 2026 (`_sample`) is sealed:** it's the natural outcome window of any past→future label built on earlier months, so it can validate query mechanics but must never inform label logic or model development.
- **Scope of these numbers:** everything measured above is observed on the `month=2026-03` slice only — directional until reproduced on another mid-panel month, not a general claim about the full panel.

## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.